In [1]:
import json
import os
import time

from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from groq import Groq

In [2]:
PROJECT_ROOT = Path.cwd().parent

ANALYSES_PATH = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "paper_analyses.csv"
)

QUERY_CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_query.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_PATH = OUTPUT_DIR / "reviewed_papers.csv"

MODEL = "llama-3.1-8b-instant"

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
with open(
    QUERY_CONFIG_PATH,
    encoding="utf-8",
) as file:
    query_config = json.load(file)

research_topic = query_config["topic"]

print(research_topic)

i want to investigate about covid-19 vaccine validation


In [5]:
df_analyses = pd.read_csv(ANALYSES_PATH)

In [6]:
reviews = []

for _, paper in df_analyses.iterrows():

    prompt = f"""
You are the Reviewer Agent in an academic research system.

The user's research topic is:

{research_topic}

Your task is to review an analysis created by another AI agent.

Original paper title:
{paper["title"]}

Original abstract:
{paper["original_abstract"]}

Analysis produced by the Analyst Agent:

Summary:
{paper["summary"]}

Main problem:
{paper["main_problem"]}

Main contribution:
{paper["main_contribution"]}

Applications:
{paper["applications"]}

Limitations:
{paper["limitations"]}

Original relevance score:
{paper["relevance_score"]}

Original relevance reason:
{paper["relevance_reason"]}

Return only a valid JSON object with exactly these fields:

{{
  "approved": true or false,
  "review_comments": "Short explanation of the review",
  "unsupported_claims": "Claims not supported by the abstract, or None",
  "corrected_relevance_score": integer from 1 to 10,
  "final_relevance_reason": "Final explanation of the paper's relevance"
}}

Relevance scoring criteria:

- 9-10: Directly addresses the user's research topic.
- 7-8: Strongly related, but not a direct match.
- 5-6: Partially related or useful only for background.
- 3-4: Weakly related.
- 1-2: Essentially unrelated.

Rules:

- Evaluate each paper independently.
- Do not default to the original score.
- Keep the original relevance score if it is well justified.
- Change it only when the abstract supports a different score.
- Use the full relevance scale from 1 to 10.
- Do not assign the same score automatically to different papers.
- approved must be true if the analysis is broadly accurate and supported.
- approved must be false if the analysis contains important unsupported claims.
- corrected_relevance_score must be an integer between 1 and 10.
- Use only the information present in the abstract.
- Return only valid JSON.
- Do not use Markdown.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={
            "type": "json_object"
        },
        temperature=0.1,
    )

    raw_review = response.choices[0].message.content

    review = json.loads(raw_review)

    review["title"] = paper["title"]
    review["authors"] = paper["authors"]
    review["published"] = paper["published"]
    review["url"] = paper["url"]
    review["original_abstract"] = paper["original_abstract"]

    review["analyst_summary"] = paper["summary"]
    review["main_problem"] = paper["main_problem"]
    review["main_contribution"] = paper["main_contribution"]
    review["applications"] = paper["applications"]
    review["limitations"] = paper["limitations"]

    review["original_relevance_score"] = paper["relevance_score"]

    reviews.append(review)

    print(f'Reviewed: {paper["title"][:70]}...')

    time.sleep(1)

Reviewed: Debiasing hazard-based, time-varying vaccine effects using vaccine-irr...


Reviewed: Efficient estimation of cumulative incidence curves via data fusion wi...


Reviewed: A simple and powerful test of vaccine waning...


Reviewed: Nonparametric bounds for vaccine effects in randomized trials...


Reviewed: Sequentially Doubly Robust Estimation of Conditional Survival Probabil...


In [7]:
df_reviews = pd.DataFrame(reviews)

In [8]:
df_reviews["approved"].value_counts()

approved
True    5
Name: count, dtype: int64

In [9]:
df_reviews

,approved,review_comments,unsupported_claims,corrected_relevance_score,final_relevance_reason,title,authors,published,url,original_abstract,analyst_summary,main_problem,main_contribution,applications,limitations,original_relevance_score
0,True,The analysis accurately captures the main cont...,None,9,The paper directly addresses the user's resear...,"Debiasing hazard-based, time-varying vaccine e...","Ethan Ashby, Dean Follmann, Holly Janes, Peter...",2025-11-19T04:15:15Z,http://arxiv.org/abs/2511.15099v1,Understanding how vaccine effectiveness (VE) c...,The paper proposes a method to mitigate bias i...,Hidden biases in time-varying vaccine effectiv...,A method to leverage vaccine-irrelevant infect...,The proposed method can be applied to provide ...,The method assumes certain conditions and may ...,9
1,True,The analysis accurately summarizes the paper's...,None,9,The paper directly addresses the user's resear...,Efficient estimation of cumulative incidence c...,"Pan Zhao, Peter B. Gilbert, Oliver Dukes, Bo Z...",2026-04-14T19:53:16Z,http://arxiv.org/abs/2604.13265v1,Refined vaccine regimens containing variant-ma...,The paper proposes methods for estimating cumu...,Estimating cumulative incidence curves for vac...,Developing efficient and multiply robust estim...,Estimating the hypothetical cumulative inciden...,"Limited to pathogens with multiple serotypes, ...",8
2,True,The analysis accurately summarizes the paper's...,None,9,The paper directly addresses the user's resear...,A simple and powerful test of vaccine waning,"Gellért Perényi, Matias Janvin, Mats J. Stensrud",2025-11-26T19:06:15Z,http://arxiv.org/abs/2511.21836v2,Determining whether vaccine efficacy wanes is ...,The paper proposes a new test to assess whethe...,Determining whether vaccine efficacy wanes ove...,"A new formal test to assess vaccine waning, wh...",The proposed test can be used in vaccine trial...,The paper does not provide a comprehensive rev...,8
3,True,The analysis accurately summarizes the paper's...,None,9,The paper directly addresses the user's resear...,Nonparametric bounds for vaccine effects in ra...,"Rachel Axelrod, Uri Obolski, Daniel Nevo",2025-10-29T09:00:30Z,http://arxiv.org/abs/2510.25296v2,Vaccine randomized trials are typically design...,The paper proposes nonparametric causal bounds...,The paper addresses the problem of estimating ...,The paper contributes nonparametric causal bou...,The proposed bounds can be applied to vaccine ...,The paper assumes the absence of unmeasured co...,7
4,True,The analysis accurately summarizes the paper's...,None,8,The paper's focus on conditional survival prob...,Sequentially Doubly Robust Estimation of Condi...,"Hongxiang Qiu, Marco Carone, Alex Luedtke, Pet...",2025-10-12T00:00:55Z,http://arxiv.org/abs/2510.10372v4,It is often of interest to study the associati...,The paper proposes a nonparametric estimator f...,Estimating conditional survival probability wi...,A sequentially doubly robust estimator for con...,"Vaccine trials, studying the association betwe...",The paper relies on the assumption that the ce...,6


In [10]:
df_reviews.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Reviews saved to:\n{OUTPUT_PATH}")

Reviews saved to:
C:\Users\JuanOrtizAlonso\ai-paper-review-agentic\data\intermediate\reviewed_papers.csv
